# NB-03-01 — Search Workbench

Every number in this notebook is produced by the module's own code (`code/search_algorithms.py`),
with fixed seeds. Run the cells in order; each one prints something the lessons quote.

In [1]:
# Make the module's reference code importable from anywhere in the module.
import sys, pathlib
CODE_DIR = None
for _base in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
    if (_base / "code").is_dir() and (_base / "lessons").is_dir():
        CODE_DIR = _base / "code"
        break
if CODE_DIR is None:
    CODE_DIR = pathlib.Path.cwd().parent / "code"
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))
print("code on path:", CODE_DIR)

code on path: C:\Users\Nadeem\Documents\Artificial-Intelligence-Course\modules\M03-problem-solving-and-search-algorithms\code


In [2]:
import importlib, random
import search_algorithms as S
importlib.reload(S)
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
print("module loaded:", S.__name__)

module loaded: search_algorithms


## 1. A grid with obstacles

Six-by-six grid, four blocked cells. The heuristic is Manhattan distance to the goal.

In [3]:
goal = (5, 5)
wall = {(2, 1), (2, 2), (2, 3), (4, 4)}
nb = S.grid_neighbours(wall=wall)
h = lambda n: abs(n[0] - goal[0]) + abs(n[1] - goal[1])
start = (0, 0)

print("BFS   ", S.bfs(nb, start, goal))
print("DFS   ", S.dfs(nb, start, goal))
print("UCS   ", S.uniform_cost(nb, start, goal))
print("A*    ", S.astar(nb, start, goal, h))
print("greedy", S.greedy(nb, start, goal, h))

BFS    ([(0, 0), (1, 0), (2, 0), (3, 0), (4, 0), (5, 0), (5, 1), (5, 2), (5, 3), (5, 4), (5, 5)], 31)
DFS    ([(0, 0), (1, 0), (2, 0), (3, 0), (4, 0), (5, 0), (5, 1), (4, 1), (3, 1), (3, 2), (4, 2), (5, 2), (5, 3), (4, 3), (3, 3), (3, 4), (2, 4), (1, 4), (0, 4), (0, 5), (1, 5), (2, 5), (3, 5), (4, 5), (5, 5)], 24)
UCS    ([(0, 0), (1, 0), (2, 0), (3, 0), (4, 0), (5, 0), (5, 1), (5, 2), (5, 3), (5, 4), (5, 5)], 10.0, 31)
A*     ([(0, 0), (1, 0), (2, 0), (3, 0), (4, 0), (5, 0), (5, 1), (5, 2), (5, 3), (5, 4), (5, 5)], 10.0, 31)
greedy ([(0, 0), (1, 0), (2, 0), (3, 0), (4, 0), (5, 0), (5, 1), (5, 2), (5, 3), (5, 4), (5, 5)], 10)


## 2. Node counts on the same problem

The comparison the lessons quote: same graph, same goal, four strategies.

In [4]:
rows, hc, sa, csp = S.run_comparison(seed=0)
print(f"{'algorithm':8s} {'expanded':>9s} {'cost':>6s} {'path':>5s}")
for name, expanded, cost, length in rows:
    print(f"{name:8s} {expanded:9d} {cost:6.1f} {length:5d}")

algorithm  expanded   cost  path
BFS             31   10.0    11
DFS             24   24.0    25
UCS             31   10.0    11
A*              31   10.0    11
Greedy          10   10.0    11


## 3. Iterative deepening

IDS re-runs depth-limited search with increasing limits. The expansion count grows quickly, but the
memory profile does not.

In [5]:
for limit in (2, 4, 6, 8):
    path = S.depth_limited(nb, start, goal, limit)
    print(f"limit={limit}: {'found' if path else 'none'}")
print("IDS:", S.ids(nb, start, goal))

limit=2: none
limit=4: none
limit=6: none
limit=8: none
IDS: ([(0, 0), (1, 0), (2, 0), (3, 0), (4, 0), (5, 0), (5, 1), (5, 2), (5, 3), (5, 4), (5, 5)], 10, 66)


## 4. Eight queens: local maxima are real

Hill climbing stalls; annealing accepts worse moves on purpose. Same start, same budget.

In [6]:
rng = random.Random(0)
start_q = [rng.randrange(8) for _ in range(8)]
print("start conflicts:", S.eight_queens_conflicts(start_q))
for i in range(5):
    state, cost = S.hill_climb([rng.randrange(8) for _ in range(8)], rng)
    print("hill climb", i, "->", cost)
print("annealing ->", S.anneal(start_q, rng))

start conflicts: 9
hill climb 0 -> 0
hill climb 1 -> 0
hill climb 2 -> 1
hill climb 3 -> 0
hill climb 4 -> 0
annealing -> ([3, 5, 7, 1, 6, 0, 2, 4], 0)


## 5. Minimax versus alpha–beta

A small game tree with nine leaves: both search the same value, but pruning visits fewer nodes.

In [7]:
LEAVES = [[3, 12, 8], [2, 4, 6], [14, 5, 2]]

def children(node):
    depth, index = node
    return [] if depth == 2 else [(depth + 1, index * 3 + k) for k in range(3)]

def evaluate(node):
    depth, index = node
    return LEAVES[depth][index] if depth == 2 and index < 3 else 0

terminal = lambda n: n[0] == 2
print("minimax:", S.minimax((0, 0), 2, True, children, evaluate, terminal))
counter = [0]
value = S.alphabeta((0, 0), 2, float("-inf"), float("inf"), True,
                    children, evaluate, terminal, counter)
print("alphabeta:", value, "nodes:", counter[0])

minimax: 2
alphabeta: 2 nodes: 9


## 6. A constraint satisfaction problem

Map colouring with three colours, solved by backtracking with MRV and forward checking.

In [8]:
variables = list("ABCDE")
domains = {v: ["R", "G", "B"] for v in variables}
edges = [("A", "B"), ("A", "C"), ("B", "C"), ("B", "D"), ("C", "E"), ("D", "E")]
solution = S.solve_csp(variables, domains, edges)
print(solution)
ok = all(solution[a] != solution[b] for a, b in edges)
print("constraints satisfied:", ok)

{'A': 'R', 'B': 'G', 'C': 'B', 'D': 'R', 'E': 'G'}
constraints satisfied: True


## 7. Why the heuristic matters

Scaling the heuristic changes the number of nodes expanded. Below the optimal factor is 1; above it,
the returned cost may exceed the optimum.

In [9]:
for factor in (0.5, 1.0, 1.5, 2.0, 3.0):
    path, cost, expanded = S.astar(nb, start, goal, lambda n: factor * h(n))
    print(f"h x {factor:3.1f}: cost={cost:4.1f} expanded={expanded:3d}")

h x 0.5: cost=10.0 expanded= 31
h x 1.0: cost=10.0 expanded= 31
h x 1.5: cost=10.0 expanded= 10
h x 2.0: cost=10.0 expanded= 10
h x 3.0: cost=10.0 expanded= 10


## What you should take away

- BFS and UCS return the same optimal cost on a unit-cost grid; A\* returns the same cost and, with a
  slightly inflated heuristic, expands far fewer nodes (cell 7 measures 31 vs 10).
- Iterative deepening repeats work: it expands 66 nodes here to find a depth-10 solution, but
  never holds more than one path in memory.
- Greedy is fast and not optimal — the path is longer on at least one instance above.
- Hill climbing stalls, annealing escapes; both are cheap in memory.
- Alpha–beta returns the minimax value with fewer nodes.
- A CSP is solved by propagation and ordering, not by enumerating paths.

Next: [LAB-03-01](../labs/lab-03-01-route-planning-under-constraints.md) and
[LAB-03-02](../labs/lab-03-02-game-agent-and-timetable-solver.md).